# Decision Tree from Scratch

In this notebook, I'm implementing a Decision Tree from scratch using
NumPy. This one is different from what I did before no weights to
learn, no distances to compare. It just asks a bunch of yes/no questions
about the data to reach a prediction.

## What is a Decision Tree?

A Decision Tree works by asking a series of yes/no questions about the
data, until it reaches a final answer. Each question splits the data
into smaller groups, and this keeps happening until we're left with
groups that are (mostly) one single class.

Let's go back to our CGPA and placement example. Instead of using a
formula like Logistic Regression, or comparing distances like KNN, a
Decision Tree might ask something like this:

Is CGPA > 7.5?
- If Yes: Has done an internship?
  - If Yes: Placed
  - If No: Not Placed
- If No: Not Placed

So if a student has CGPA 8.5 and has done an internship, we just follow
the tree down: CGPA > 7.5? Yes. Internship? Yes. That leads us to
"Placed". If another student has CGPA 6.0, we go: CGPA > 7.5? No. That
directly leads to "Not Placed", without even checking internship status.

A few terms worth knowing here:
- **Root node** - the very first question at the top of the tree
- **Node** - any point in the tree where a question is being asked
- **Leaf node** - the final boxes where we actually get an answer
  (Placed / Not Placed), with no more questions after that
- **Branch** - the path connecting one node to the next, based on the
  answer (yes/no)

The tricky part is figuring out *which* question to ask at each step,
and *where* exactly to draw the line (like why CGPA > 7.5 and not 7.0 or
8.0). That's what the math in the next few sections is actually for.

## How Does the Tree Pick the Best Question?

The tree isn't just picking random questions to ask. It's trying to find
the question that splits the students into the "cleanest" possible
groups.

By clean, I mean something like this - if we split on "CGPA > 7.5" and
almost everyone above that got placed, and almost everyone below didn't,
that's a really good question. But if both sides end up with a random
mix of placed and not-placed students, that question isn't helping us
much.

So we need some way to actually measure how "mixed up" a group is. This
is where something called **impurity** comes in. There are two common
ways to calculate it - **Gini Impurity** and **Entropy**. We'll go
through both.

## Gini Impurity

Gini Impurity is just a way to measure how mixed up a group is. Here's
the formula:

$$Gini(S) = 1 - \sum_{i=1}^{K} p_i^2$$

$S$ is the group of samples we're looking at, $K$ is how many classes
there are, and $p_i$ is the fraction of each class in that group.

Let's just plug in some numbers to see how this works. Say we have 10
students - 8 placed, 2 not placed.

$$p_{placed} = 8/10 = 0.8, \quad p_{not\ placed} = 2/10 = 0.2$$

$$Gini(S) = 1 - (0.8^2 + 0.2^2) = 1 - 0.68 = 0.32$$

Now if the group was perfectly pure, say all 10 were placed:

$$Gini(S) = 1 - (1^2 + 0^2) = 0$$

So Gini = 0 means the group is fully pure, only one class in there. The
more mixed the group, the higher the number goes, maxing out at 0.5 when
it's a perfect 50-50 split.

## How Do We Actually Pick the Best Split?

Once we know how to measure impurity, we need a way to compare different
splits and figure out which one is actually good. That's what
Information Gain does:

$$IG(S,A) = Impurity(S) - \sum_{j} \frac{N_j}{N} \cdot Impurity(S_j)$$

$S$ is the data before we split it, $A$ is whatever feature we're
splitting on, $S_j$ is each smaller group after the split, $N_j$ is how
many samples are in that group, and $N$ is the total number of samples
before splitting.

Basically we're just taking the impurity before the split, and
subtracting the impurity after the split (averaged based on group size).
If a split does a good job separating placed vs not placed, the impurity
after will be small, so we end up subtracting a small number, which
means the gain is high.

To actually build the tree, we just try a bunch of splits, calculate the
gain for each, and go with whichever one gives the highest gain.

## When Does the Tree Stop Growing?

Here's a problem - if we let the tree keep asking questions forever, it
will eventually split the data so much that each tiny group has just one
or two students in it. At that point, the tree isn't really learning any
useful pattern anymore, it's basically just memorizing every single
student individually.

This causes a big issue called **overfitting**. The tree ends up doing
great on the training data (because it literally memorized it), but it
performs badly on any new student it hasn't seen before, since it never
actually learned a general rule.

So we need to tell the tree when to stop splitting. A few common ways to
do this are:

- **Max depth** - just limit how many questions deep the tree can go.
  For example, if max depth is 3, the tree can only ask 3 levels of
  questions before it has to stop and make a final decision.
- **Minimum samples to split** - don't allow the tree to split a group
  any further if it has too few samples left (like less than 5
  students). Splitting a tiny group doesn't really teach the model
  anything useful anyway.
- **Stop if already pure** - if a group's Gini impurity is already 0
  (meaning everyone in that group is the same class), there's no point
  splitting it further since there's nothing left to separate.

For our implementation, we'll keep it simple and just use a max depth
limit, since it's the easiest one to control and understand.

## Putting It All Together


1. Start with all the data at the top (this is the root)
2. Try out every possible split - every feature, every possible
   threshold value
3. For each one, calculate the information gain
4. Pick whichever split gave the highest gain
5. Split the data into two smaller groups based on that
6. Now do the exact same thing again, but separately for each of those
   two smaller groups
7. Keep repeating this until we hit a stopping point - either the group
   is already pure, or we've hit our max depth
8. Once we stop at a group, that becomes a leaf, and the prediction is
   just whatever class shows up the most in that group

When we want to predict something new, we just start at the root and
keep following the questions (yes/no) until we land on a leaf, and
that's our answer.